In [ ]:
import os
from typing import Any

import pandas as pd
import requests
from IPython.display import display


def fetch_all_users(
    base_url: str | None = None,
    access_token: str | None = None,
    endpoint: str = "/api/admin/users",
    timeout: int = 30,
) -> list[dict[str, Any]]:
    """Fetch all users from the API.

    Expects either:
    - a list response: [{...}, {...}]
    - or an object with a users key: {"users": [{...}]}

    Args:
        base_url: API base URL, defaults to API_BASE_URL env var.
        access_token: JWT access token, defaults to ACCESS_TOKEN env var.
        endpoint: Relative endpoint for listing users.
        timeout: Request timeout in seconds.
    """
    base_url = base_url or os.getenv("API_BASE_URL", "http://localhost:8000")
    access_token = access_token or os.getenv("ACCESS_TOKEN")

    if not access_token:
        raise ValueError(
            "Missing access token. Set ACCESS_TOKEN env var or pass access_token="
        )

    url = f"{base_url.rstrip('/')}{endpoint}"
    headers = {
        "Authorization": f"Bearer {access_token}",
        "Accept": "application/json",
    }

    response = requests.get(url, headers=headers, timeout=timeout)
    response.raise_for_status()

    data = response.json()
    if isinstance(data, list):
        return data
    if isinstance(data, dict) and isinstance(data.get("users"), list):
        return data["users"]

    raise ValueError(f"Unexpected response shape from {url}: {type(data).__name__}")


def display_all_users(
    base_url: str | None = None,
    access_token: str | None = None,
    endpoint: str = "/api/admin/users",
) -> pd.DataFrame:
    """Fetch users and display them as a DataFrame in the notebook."""
    users = fetch_all_users(
        base_url=base_url,
        access_token=access_token,
        endpoint=endpoint,
    )

    df = pd.DataFrame(users)

    # Move common identity columns to front if present.
    front_cols = [c for c in ["id", "name", "email", "is_active", "created_at"] if c in df.columns]
    other_cols = [c for c in df.columns if c not in front_cols]
    df = df[front_cols + other_cols] if not df.empty else df

    display(df)
    print(f"Total users: {len(df)}")
    return df


# Example usage:
# os.environ["API_BASE_URL"] = "https://advicetools-production.up.railway.app"
# os.environ["ACCESS_TOKEN"] = "<your-admin-access-token>"
# display_all_users()
